# Data Exploration Notebook

This notebook explores the image data used in the MLOps project.

In [ ]:
# Import necessary libraries
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import cv2

# Add the src directory to the path so we can import our modules
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'src'))

from segmentation import contour_detection, find_bounding_boxes, find_main_bounding_box
from image_processing import contour, apply_clahe
from utils import calcul_dev, calculate_ch, calculate_entropy_variation_rhythm

## Load and Analyze Images

First, let's load some images from our dataset and analyze their characteristics.

In [ ]:
# Set paths
data_dir = os.path.join(os.path.dirname(os.getcwd()), 'data', 'inputs')

# List image files
image_files = [f for f in os.listdir(data_dir) 
               if os.path.isfile(os.path.join(data_dir, f)) and 
               f.lower().endswith(('.png', '.jpg', '.jpeg'))]

print(f"Found {len(image_files)} images")
print(f"Image files: {image_files[:5]}..." if len(image_files) > 5 else f"Image files: {image_files}")

In [ ]:
# Load a sample image
if image_files:
    sample_image_path = os.path.join(data_dir, image_files[0])
    sample_image = Image.open(sample_image_path)
    sample_array = np.array(sample_image)
    
    # Display the image
    plt.figure(figsize=(10, 8))
    plt.imshow(sample_array)
    plt.title(f"Sample Image: {image_files[0]}")
    plt.axis('off')
    plt.show()
    
    # Print image information
    print(f"Image shape: {sample_array.shape}")
    print(f"Image type: {sample_array.dtype}")
    print(f"Min value: {np.min(sample_array)}")
    print(f"Max value: {np.max(sample_array)}")
    print(f"Mean value: {np.mean(sample_array)}")
    print(f"Standard deviation: {np.std(sample_array)}")
else:
    print("No images found in the data directory.")

## Apply Image Processing

Let's apply our image processing pipeline to see how it transforms the images.

In [ ]:
# Process a sample image
if 'sample_array' in locals():
    # Apply contour detection
    contours = contour(sample_array)
    
    # Display the contours
    plt.figure(figsize=(10, 8))
    plt.imshow(contours, cmap='gray')
    plt.title("Contour Detection")
    plt.axis('off')
    plt.show()
    
    # Find bounding boxes
    boxes = find_bounding_boxes(contours)
    main_box = find_main_bounding_box(boxes)
    
    # Draw bounding boxes
    vis_image = sample_array.copy()
    for box in boxes:
        cv2.rectangle(vis_image, 
                     (box.x, box.y), 
                     (box.x + box.width, box.y + box.height), 
                     (0, 255, 0), 2)
    
    if main_box:
        cv2.rectangle(vis_image, 
                     (main_box.x, main_box.y), 
                     (main_box.x + main_box.width, main_box.y + main_box.height), 
                     (255, 0, 0), 3)
    
    # Display the image with bounding boxes
    plt.figure(figsize=(10, 8))
    plt.imshow(vis_image)
    plt.title("Bounding Boxes")
    plt.axis('off')
    plt.show()
    
    # Crop the image
    if main_box:
        cropped = sample_image.crop((
            main_box.x, 
            main_box.y, 
            main_box.x + main_box.width, 
            main_box.y + main_box.height
        ))
        cropped_array = np.array(cropped)
        
        # Display the cropped image
        plt.figure(figsize=(10, 8))
        plt.imshow(cropped_array)
        plt.title("Cropped Image")
        plt.axis('off')
        plt.show()
        
        # Apply CLAHE
        enhanced = apply_clahe(cropped_array)
        
        # Display the enhanced image
        plt.figure(figsize=(10, 8))
        plt.imshow(enhanced)
        plt.title("Enhanced Image (CLAHE)")
        plt.axis('off')
        plt.show()

## Extract Features

Now let's extract features from the processed image.

In [ ]:
# Extract features
if 'enhanced' in locals():
    # RGB standard deviation
    rgb_features = calcul_dev(enhanced)
    print(f"RGB features: {rgb_features}")
    
    # Create a bar chart of RGB features
    plt.figure(figsize=(8, 6))
    plt.bar(['R', 'G', 'B'], rgb_features)
    plt.title('RGB Standard Deviations')
    plt.ylabel('Standard Deviation')
    plt.show()
    
    # Calculate convex hull ratio
    ch_ratio = calculate_ch(enhanced)
    print(f"Convex hull ratio: {ch_ratio}")
    
    # Calculate entropy variation
    entropy_map = calculate_entropy_variation_rhythm(enhanced[:, :, 0])
    
    # Display entropy map
    plt.figure(figsize=(10, 8))
    sns.heatmap(entropy_map, cmap='viridis')
    plt.title('Entropy Variation Map')
    plt.show()

## Analyze All Images

Let's process all images and analyze their features.

In [ ]:
# Process all images and collect features
features_list = []
file_names = []

for image_file in image_files[:10]:  # Limit to 10 images for this notebook
    try:
        image_path = os.path.join(data_dir, image_file)
        
        # Open the image
        with Image.open(image_path) as img:
            im_arr = np.array(img)
        
        # Apply contour detection
        contours = contour(im_arr)
        
        # Find bounding boxes
        boxes = find_bounding_boxes(contours)
        main_box = find_main_bounding_box(boxes)
        
        if main_box:
            # Crop the image
            cropped = im_arr[main_box.y:main_box.y+main_box.height, main_box.x:main_box.x+main_box.width]
            
            # Apply CLAHE
            enhanced = apply_clahe(cropped)
            
            # Extract features
            features = calcul_dev(enhanced)
            
            # Add to lists
            features_list.append(features)
            file_names.append(image_file)
            
    except Exception as e:
        print(f"Error processing {image_file}: {str(e)}")

# Convert to DataFrame
features_df = pd.DataFrame(features_list, columns=['R', 'G', 'B'], index=file_names)
features_df.head()

In [ ]:
# Visualize feature distributions
plt.figure(figsize=(12, 6))

plt.subplot(1, 3, 1)
sns.histplot(features_df['R'], kde=True)
plt.title('R Channel Distribution')

plt.subplot(1, 3, 2)
sns.histplot(features_df['G'], kde=True)
plt.title('G Channel Distribution')

plt.subplot(1, 3, 3)
sns.histplot(features_df['B'], kde=True)
plt.title('B Channel Distribution')

plt.tight_layout()
plt.show()

# Feature correlations
plt.figure(figsize=(8, 6))
sns.heatmap(features_df.corr(), annot=True, cmap='coolwarm')
plt.title('Feature Correlations')
plt.show()

## Conclusions

Based on our exploration, we can see that the image processing pipeline effectively:
1. Detects contours in the images
2. Identifies the main object with bounding boxes
3. Crops the image to focus on the main object
4. Enhances the image using CLAHE
5. Extracts meaningful features for classification

The RGB standard deviation features show distinct patterns that could be useful for classification tasks.